# Semanas 3-4 - WVC 2026 - SimCLR-indomain (treino, convergencia, congelamento)

**Escopo (do §11 do protocolo):** treinar o **SimCLR-indomain**, registrar convergencia,
congelar features. Este e o objeto **controlavel** das H1-H3 (voce define as augmentations),
e o unico braco com **risco de execucao real** (guarda-c do gate e da Semana 2).

**Timebox rigido `[§11]`:** ao fim da Semana 4, o que houver e o SSL-indomain do paper,
**reportado como e**. Nao estender. Este notebook implementa o timebox em codigo (parada por
wall-clock + selecao do melhor checkpoint), para que a decisao "parar" nao dependa de forca
de vontade a 3 dias do deadline.

**Correcao de OOM (revisao 1) `[novo]`:** o treino deu `OutOfMemoryError` no HPC. Como no SimCLR
o **batch e a fonte de negativos**, a ordem de correcao preserva o batch antes de mexer nele:
(1) `expandable_segments` anti-fragmentacao, (2) **gradient checkpointing** no backbone (troca
compute por memoria, mantem o batch), (3) `channels_last` + AMP, (4) **auto-fit** que so reduz o
batch em ultimo caso **e registra** que perdeu negativos. `resnet18`/DDP ficam como decisao
seguinte (a de batch pequeno degrada a feature por *memoria*, nao por metodo - e isso entra no
registro para nao contaminar a leitura da barra).

---

## O que estas duas semanas provam - e o que NAO provam (ler antes de rodar)

Herdado dos registros do gate e da Semana 2, mais o que e novo aqui:

- **(a) Numeros internos NAO sao numeros de paper.** Split same-cohort do 100k tem vazamento
  same-slide provavel. So o 7k entra em tabela de desempenho. Aqui o interno mede **movimento
  relativo do gap sob controle** (H1/H2/H3 no eixo SimCLR) - diagnostico, nao resultado.

- **(f) A variancia honesta do SimCLR e ACROSS-PRETRAIN, nao within-probe `[CRITICO desta fase]`.**
  Repetir 5 seeds de probe sobre **um unico** SimCLR congelado subreporta a variancia, porque
  ignora a variancia de seed de **pre-treino**. Este e literalmente o formato do erro documentado
  (IC apertado que nao e honesto). O notebook separa as duas variancias e reporta a de pre-treino
  como a valida. Se so 1 pre-treino couber no timebox, o paper **declara** que a variancia do SSL
  esta subcaracterizada - nao copia o IC estreito do gate/Semana 2 para o SimCLR.

- **(vazamento transdutivo) `[novo, especifico de SSL in-domain]`.** O SimCLR pre-treina **so em
  `idx_train_pool` (80k)**, nunca em `idx_eval_int` (20k). Se pre-treinasse nos 100k inteiros,
  veria (sem rotulo) as imagens onde H2/H3 sao medidos no split interno - boost transdutivo que
  LBP/Phikon nao tem ali, enviesando os diagnosticos. **Desvio registrado do §6** ("pre-treina no
  pool inteiro"): custa ~20% dos dados nao-rotulados, compra medicao interna limpa de H2/H3.
  Registrar com data no `.md` da semana.

- **(stain) `[bloqueia comparabilidade]`.** O SimCLR le da MESMA lista de caminhos (`paths_100k`,
  ordem canonica) e da MESMA condicao de stain que LBP/Fractal/Phikon. Se `NCT_DIR` nao continha
  Macenko (item aberto da Semana 2), o problema e herdado igualmente por todos os bracos - mas
  precisa ser **confirmado upstream** antes de travar Metodos. O notebook nao normaliza; assume
  que `NCT_DIR` ja e a condicao do nucleo.

- **(H3 e o que faz o SimCLR merecer a vaga - e o primeiro corte do timebox).** Sem a ablacao
  H3-a (ligar/desligar a invariancia de escala/cor que o LBP nao modela), o SimCLR vira "um Phikon
  mais fraco". O notebook prioriza: (1) 1 SimCLR full-aug utilizavel -> (2) seeds de pre-treino
  para variancia -> (3) ablacao H3-a. Corta de tras para frente.

- **(fallback §10-3).** Se o SimCLR sair mediocre (nao limpa a barra do classico com folga), isso
  e **achado reportavel** (§4 do protocolo), nao buraco. A historia recua para "FM-ref congelado
  vs. classico" com SimCLR como contexto secundario. **A escrita nao pode sugerir que estas
  semanas provaram algo sobre o SSL alem do que a barra interna mostrar.**


## 0. Configuracao (edite so esta celula)

Dois nos GPU. Os knobs de custo (backbone, batch, epocas, seeds de pre-treino, wall-clock) sao
onde o timebox vive. Defaults deliberadamente **modestos** - a guarda-c ja sanciona um SimCLR de
orcamento modesto como achado reportavel, entao e melhor terminar modesto do que nao terminar.

**Memoria `[revisao OOM]`:** `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` e setado no topo
desta celula, **antes** do import de torch (senao o alocador ja inicializou e ignora). Idealmente
tambem exporte no script do job (`export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`).
`GRAD_CKPT=True` liga o checkpointing do backbone.

In [ ]:
import os
# anti-fragmentacao do alocador CUDA. TEM que vir antes de 'import torch'.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from pathlib import Path

# --- Dados (MESMA ordem/condicao dos outros bracos) --------------------------
NCT_DIR      = Path("./NCT-CRC-HE-100K")     # <-- EDITE (mesma pasta da Semana 2)
CRC_VAL_DIR  = Path("./CRC-VAL-HE-7K")       # <-- EDITE
# Caches da Semana 2 (LBP/Fractal/Phikon) para o head-to-head interno. None -> pula comparacao.
WEEK2_CACHE_DIR = None                        # <-- EDITE (aponta p/ ./cache_semana2 do notebook anterior)

CACHE_DIR = Path("./cache_semana34"); CACHE_DIR.mkdir(exist_ok=True, parents=True)
OUT_DIR   = Path("./out_semana34");   OUT_DIR.mkdir(exist_ok=True, parents=True)
CKPT_DIR  = Path("./ckpt_simclr");    CKPT_DIR.mkdir(exist_ok=True, parents=True)

# --- Split / protocolo (TRAVADO - herdado do gate e da Semana 2) -------------
INTERNAL_SPLIT_SEED = 20260816
INTERNAL_TEST_SIZE  = 0.20
CLASSES = ["ADI","BACK","DEB","LYM","MUC","MUS","NORM","STR","TUM"]   # ordem fixa

# --- Guarda de vazamento transdutivo (desvio REGISTRADO do §6) ---------------
PRETRAIN_ON_TRAIN_POOL_ONLY = True

# --- SimCLR: arquitetura -----------------------------------------------------
BACKBONE   = "resnet50"     # "resnet18" (512-d, mais estavel/rapido, timebox-safe) | "resnet50" (2048-d)
PROJ_DIM   = 128
TEMP       = 0.2
GRAD_CKPT  = True           # gradient checkpointing no backbone (troca compute por memoria; fix de OOM)

# --- SimCLR: otimizacao (knobs de TIMEBOX) -----------------------------------
BATCH        = 256          # ponto de PARTIDA; o auto-fit reduz se nao couber (e registra a perda de negativos)
MIN_BATCH    = 32           # piso do auto-fit; abaixo disso -> resnet18/DDP
EPOCHS_CAP   = 200
WARMUP_EPOCH = 10
BASE_LR      = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 8
AMP          = True

# --- TIMEBOX em wall-clock ---------------------------------------------------
WALLCLOCK_HOURS_PER_RUN = 12.0

# --- Variancia HONESTA: seeds de PRE-TREINO (guarda-f) -----------------------
PRETRAIN_SEEDS = [0, 1, 2]

# --- Augmentations e H3-a ----------------------------------------------------
AUG_CONFIG = "full"              # "full" | "ablated_no_scale_color"

# --- Convergencia (criterio PRE-REGISTRADO) ----------------------------------
MONITOR_EVERY      = 5
MONITOR_PER_CLASS  = 40
KNN_K              = 20
PLATEAU_DELTA      = 0.3
PLATEAU_PATIENCE   = 4
COLLAPSE_STD_MIN   = 1e-3

# --- Probe interno (reusa o aparato da Semana 2) -----------------------------
LABEL_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 1.00]
LABEL_SEED_BASE = 0
N_LABEL_SEEDS   = 5
LABEL_SEEDS     = [LABEL_SEED_BASE + i for i in range(N_LABEL_SEEDS)]
COMMON_D        = 32
D_SWEEP         = [2, 4, 8, 16, 24, 32, 48, 64]
D_SWEEP_FRACTIONS = [0.05, 1.00]

# --- Alvo de avaliacao -------------------------------------------------------
EVAL_TARGET = "internal"

import numpy as np, random
GLOBAL_SEED = 20260817
np.random.seed(GLOBAL_SEED); random.seed(GLOBAL_SEED)
print("Config OK.",
      "| BACKBONE =", BACKBONE, "| BATCH =", BATCH, "| GRAD_CKPT =", GRAD_CKPT,
      "| PRETRAIN_SEEDS =", PRETRAIN_SEEDS, "| AUG =", AUG_CONFIG,
      "| pool =", ("train_pool_80k" if PRETRAIN_ON_TRAIN_POOL_ONLY else "full_100k"))
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))


In [ ]:
import json, hashlib, time, warnings, math
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, f1_score
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint_sequential

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU  = torch.cuda.device_count()
print("torch", torch.__version__, "| device", DEVICE, "| GPUs", N_GPU)


## 1. Indexacao + split (identicos ao gate e a Semana 2)

A lista `(caminho, rotulo)` define a **ordem canonica** de todas as matrizes de feature. As
features do SimCLR serao salvas alinhadas a esta ordem, para caírem no MESMO aparato de probe/PCA
da Semana 2. Reusa `INTERNAL_SPLIT_SEED` e `INTERNAL_TEST_SIZE` -> o `idx_eval_int` e o mesmo
conjunto de eval dos outros bracos.

In [ ]:
def index_folder(root: Path, classes):
    paths, y = [], []
    for ci, c in enumerate(classes):
        cdir = root / c
        if not cdir.is_dir():
            raise FileNotFoundError(f"Subpasta de classe ausente: {cdir}")
        files = sorted(p for p in cdir.iterdir()
                       if p.suffix.lower() in {".tif",".tiff",".png",".jpg",".jpeg"})
        paths += files; y += [ci]*len(files)
    return np.array([str(p) for p in paths]), np.array(y, dtype=np.int64)

paths_100k, y_100k = index_folder(NCT_DIR, CLASSES)
N = len(paths_100k)
print(f"NCT-CRC-HE-100K indexado: {N} imagens")
print("Contagem por classe:", dict(zip(CLASSES, np.bincount(y_100k, minlength=len(CLASSES)))))
assert N > 0, "Nenhuma imagem encontrada - confira NCT_DIR."

idx_all = np.arange(N)
idx_train_pool, idx_eval_int = train_test_split(
    idx_all, test_size=INTERNAL_TEST_SIZE, stratify=y_100k, random_state=INTERNAL_SPLIT_SEED)
idx_train_pool = np.sort(idx_train_pool); idx_eval_int = np.sort(idx_eval_int)
y_train_pool = y_100k[idx_train_pool]
print(f"Split interno: train_pool={len(idx_train_pool)}  eval_int={len(idx_eval_int)}")

if PRETRAIN_ON_TRAIN_POOL_ONLY:
    idx_pretrain = idx_train_pool.copy()
    print(f"[guarda transdutiva] pre-treino SO em train_pool: {len(idx_pretrain)} imagens. "
          f"eval_int ({len(idx_eval_int)}) intocado pelo SSL.")
else:
    idx_pretrain = idx_all.copy()
    print(f"[AVISO] pre-treino nos 100k inteiros: {len(idx_pretrain)} imagens. Transdutivo no interno.")

def label_subsample(frac, seed):
    if frac >= 1.0:
        return np.arange(len(idx_train_pool))
    sub, _ = train_test_split(np.arange(len(idx_train_pool)), train_size=frac,
                              stratify=y_train_pool, random_state=seed)
    return np.sort(sub)

_a = label_subsample(0.05, 3); _b = label_subsample(0.05, 3)
assert np.array_equal(_a, _b) and not np.array_equal(_a, label_subsample(0.05, 4))
print("Subamostragem deterministica e seed-sensivel: OK.")


### 1.1 Conjuntos do kNN-monitor (para SELECAO de checkpoint, nao para numero de paper)

Pequenos subconjuntos rotulados **de dentro do `idx_train_pool`**, disjuntos entre si. Servem so
para monitorar convergencia e escolher o melhor checkpoint. `idx_eval_int` continua limpo.

In [ ]:
def _carve_monitor(idx_pool, y_pool, per_class, seed=0):
    rng = np.random.default_rng(seed)
    tr, ev = [], []
    for ci in range(len(CLASSES)):
        pos = np.where(y_pool == ci)[0]
        rng.shuffle(pos)
        take = min(2*per_class, len(pos)); half = take // 2
        tr += list(idx_pool[pos[:half]]); ev += list(idx_pool[pos[half:2*half]])
    return np.array(sorted(tr)), np.array(sorted(ev))

mon_tr_idx, mon_ev_idx = _carve_monitor(idx_train_pool, y_train_pool, MONITOR_PER_CLASS)
print(f"kNN-monitor: train={len(mon_tr_idx)}  eval={len(mon_ev_idx)} (disjuntos, do train_pool)")


## 2. Augmentations - conjunto full e ablacao H3-a

**Design de H3-a `[o ponto que faz o SimCLR ser o objeto controlavel]`:** a ablacao remove
exatamente a invariancia que o **LBP nao modela** - escala (variacao do RandomResizedCrop) e cor
(color jitter + grayscale). **Rotacao 90 e flip ficam nos DOIS conjuntos** (o LBP-ri ja e
rotacao-invariante -> isola escala/cor como a unica diferenca). Blur nos dois (nao e escala nem
cor). Color jitter interage com o Macenko de proposito - e o ponto de H3-a.

In [ ]:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

class RandomRot90:
    def __call__(self, img):
        k = int(torch.randint(0, 4, (1,)).item())
        return img.rotate(90*k) if k else img

def _blur():
    return T.RandomApply([T.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0))], p=0.5)

def build_ssl_transform(aug_config):
    head_full = T.RandomResizedCrop(224, scale=(0.2, 1.0))           # ESCALA (full)
    head_abl  = T.Compose([T.Resize(224), T.CenterCrop(224)])        # sem escala (ablado)
    color = T.Compose([T.RandomApply([T.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
                       T.RandomGrayscale(p=0.2)])                     # COR (full)
    spatial = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(), RandomRot90()])  # DOIS
    if aug_config == "full":
        pipe = T.Compose([head_full, spatial, color, _blur(),
                          T.ToTensor(), T.Normalize(NORM_MEAN, NORM_STD)])
    elif aug_config == "ablated_no_scale_color":
        pipe = T.Compose([head_abl, spatial, _blur(),
                          T.ToTensor(), T.Normalize(NORM_MEAN, NORM_STD)])
    else:
        raise ValueError(aug_config)
    return pipe

eval_transform = T.Compose([T.Resize(224), T.CenterCrop(224),
                            T.ToTensor(), T.Normalize(NORM_MEAN, NORM_STD)])
print("Transforms prontos. AUG_CONFIG atual:", AUG_CONFIG)


In [ ]:
class TwoViewDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths; self.t = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.t(img), self.t(img)

class EvalDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths; self.t = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.t(Image.open(self.paths[i]).convert("RGB"))
print("datasets ok")


## 3. Modelo (encoder + cabeca de projecao) e NT-Xent - com gradient checkpointing

**Fix de OOM `[revisao 1]`:** o backbone e quebrado em `stem` + `stages` (as 4 layers do ResNet), e
o forward das `stages` passa por `checkpoint_sequential(..., use_reentrant=False)` quando em treino.
Isso **nao recomputa** ativacoes intermediarias no forward (recomputa no backward), cortando o pico
de memoria **sem tocar no batch** - portanto sem perder negativos do SimCLR. Em eval/extracao o
checkpointing fica desligado (nao ha backward). Features de downstream = **saida do encoder `h`**
(padrao SimCLR); a projecao e descartada no congelamento.

In [ ]:
class SimCLRNet(nn.Module):
    def __init__(self, backbone_name, proj_dim, grad_ckpt=True):
        super().__init__()
        base = torchvision.models.__dict__[backbone_name](weights=None)
        self.feat_dim = base.fc.in_features
        self.stem    = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.stages  = nn.Sequential(base.layer1, base.layer2, base.layer3, base.layer4)
        self.avgpool = base.avgpool
        self.grad_ckpt = grad_ckpt
        self.proj = nn.Sequential(nn.Linear(self.feat_dim, self.feat_dim), nn.ReLU(inplace=True),
                                  nn.Linear(self.feat_dim, proj_dim))
    def encode(self, x):
        x = self.stem(x)
        if self.grad_ckpt and self.training:
            x = checkpoint_sequential(self.stages, 4, x, use_reentrant=False)  # economia de memoria
        else:
            x = self.stages(x)
        return torch.flatten(self.avgpool(x), 1)
    def forward(self, x):
        h = self.encode(x)
        return h, F.normalize(self.proj(h), dim=1)

def nt_xent(z, temp):
    # z: (2N, d) L2-normalizado; views empilhadas [view1(0..N-1); view2(N..2N-1)]
    N2 = z.shape[0]; Nb = N2 // 2
    sim = (z @ z.t()) / temp
    sim.fill_diagonal_(float("-inf"))                        # remove auto-similaridade
    pos = (torch.arange(N2, device=z.device) + Nb) % N2       # positivo de k e k+N (mod 2N)
    return F.cross_entropy(sim, pos)

# self-test do loss
with torch.no_grad():
    Nb, d = 8, 16
    base = F.normalize(torch.randn(Nb, d), dim=1)
    z_good = F.normalize(torch.cat([base, base + 1e-3*torch.randn_like(base)], 0), dim=1)
    z_rand = F.normalize(torch.randn(2*Nb, d), dim=1)
    lg, lr = nt_xent(z_good, 0.2).item(), nt_xent(z_rand, 0.2).item()
assert lg < lr, "NT-Xent com sinal invertido - revisar indices de positivo."
print(f"SimCLRNet(grad_ckpt) + NT-Xent OK (alinhado {lg:.3f} << aleatorio {lr:.3f})")


## 4. Monitores de convergencia (kNN-monitor + deteccao de colapso)

- **kNN-monitor:** classifica `mon_ev` por kNN coseno sobre features do encoder de `mon_tr`. Proxy
  de *qualidade da representacao* - a loss cair NAO garante feature boa. Define convergencia e
  escolhe checkpoint.
- **Colapso:** std medio por dimensao das features L2-normalizadas. Colapso -> std ~ 0.

`encode()` usa `channels_last` e batch menor (128) - eval nao tem grad, entao nao precisa do batch
de treino e evita OOM aqui tambem.

In [ ]:
@torch.no_grad()
def encode(model, paths, batch=128):
    model.eval()
    ds = EvalDataset(paths, eval_transform)
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    m = model.module if isinstance(model, nn.DataParallel) else model
    feats = []
    for xb in dl:
        xb = xb.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        feats.append(m.encode(xb).float().cpu().numpy())
    return np.concatenate(feats)

def knn_monitor(model):
    Xtr = encode(model, paths_100k[mon_tr_idx]); ytr = y_100k[mon_tr_idx]
    Xev = encode(model, paths_100k[mon_ev_idx]); yev = y_100k[mon_ev_idx]
    Ftr = Xtr / (np.linalg.norm(Xtr, axis=1, keepdims=True) + 1e-9)
    Fev = Xev / (np.linalg.norm(Xev, axis=1, keepdims=True) + 1e-9)
    sim = Fev @ Ftr.T
    idx = np.argsort(-sim, axis=1)[:, :KNN_K]
    pred = []
    for row in idx:
        vals, cnts = np.unique(ytr[row], return_counts=True)
        pred.append(vals[np.argmax(cnts)])
    ba = 100*balanced_accuracy_score(yev, np.array(pred))
    std = float(Fev.std(axis=0).mean())
    return ba, std
print("monitores prontos")


## 5. Loop de treino com TIMEBOX, criterio de convergencia e auto-fit de batch (OOM)

**Criterio de convergencia (registrado ANTES de rodar):** *convergido* se o kNN-monitor entrar em
**plato** (variacao < `PLATEAU_DELTA` por `PLATEAU_PATIENCE` avaliacoes) **e** o std ficar acima de
`COLLAPSE_STD_MIN`. Se `EPOCHS_CAP` ou o **wall-clock** chegar antes, congela no **melhor checkpoint
por kNN-monitor** (timebox). Status ("plato"|"timebox"|"colapso"|"epochs_cap") vai para o registro.

**Auto-fit de batch `[fix OOM]`:** `train_simclr` tenta `BATCH`; em `OutOfMemoryError` ele **reduz
pela metade e retenta** ate `MIN_BATCH`, imprimindo um AVISO de que perdeu negativos (a REGISTRAR).
Como grad checkpointing + `expandable_segments` ja cortam o pico, o esperado e nem precisar reduzir
com ResNet-50/batch 256 em GPU de 16 GB+. Se cair ate `MIN_BATCH`, e sinal de trocar para
`resnet18` ou DDP - decisao registrada, nao silenciosa.

In [ ]:
def cosine_warmup_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * base_lr * (1 + math.cos(math.pi * p))

def _try_train(pretrain_seed, aug_config, tag, batch):
    torch.manual_seed(pretrain_seed); np.random.seed(pretrain_seed)
    model = SimCLRNet(BACKBONE, PROJ_DIM, grad_ckpt=GRAD_CKPT).to(DEVICE).to(memory_format=torch.channels_last)
    if N_GPU > 1: model = nn.DataParallel(model)
    opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)

    ds = TwoViewDataset(paths_100k[idx_pretrain], build_ssl_transform(aug_config))
    dl = DataLoader(ds, batch_size=batch, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=True, drop_last=True)
    total_steps = EPOCHS_CAP * len(dl)

    hist, best_ba, best_ep, plateau_ctr, status = [], -1.0, -1, 0, "running"
    t_start, gstep = time.time(), 0
    best_path = CKPT_DIR / f"simclr_{tag}_best.pt"

    for ep in range(EPOCHS_CAP):
        model.train(); ep_loss = 0.0
        for v1, v2 in dl:
            lr = cosine_warmup_lr(gstep, total_steps, WARMUP_EPOCH*len(dl), BASE_LR)
            for g in opt.param_groups: g["lr"] = lr
            x = torch.cat([v1, v2], 0).to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=AMP):
                _, z = model(x); loss = nt_xent(z, TEMP)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            ep_loss += loss.item(); gstep += 1

        elapsed_h = (time.time() - t_start) / 3600.0
        if ((ep+1) % MONITOR_EVERY == 0) or (ep == EPOCHS_CAP-1):
            ba, std = knn_monitor(model)
            hist.append(dict(epoch=ep+1, loss=ep_loss/len(dl), knn_ba=ba, std=std, lr=lr, elapsed_h=elapsed_h))
            improved = ba > best_ba + 1e-9
            if improved:
                best_ba, best_ep = ba, ep+1
                sd = (model.module if isinstance(model, nn.DataParallel) else model).state_dict()
                torch.save({"state_dict": sd, "epoch": ep+1, "knn_ba": ba, "aug": aug_config,
                            "seed": pretrain_seed, "backbone": BACKBONE, "batch": batch}, best_path)
            plateau_ctr = plateau_ctr + 1 if (ba - best_ba) < PLATEAU_DELTA - 1e-9 and not improved else 0
            print(f"  [{tag}|b{batch}] ep {ep+1:3d} loss {ep_loss/len(dl):.3f} knn_ba {ba:5.2f} "
                  f"std {std:.4f} {elapsed_h:.1f}h (best {best_ba:.2f}@{best_ep})")
            if std < COLLAPSE_STD_MIN:
                status = "colapso"; print(f"  [{tag}] COLAPSO (std<{COLLAPSE_STD_MIN}). Parando."); break
            if plateau_ctr >= PLATEAU_PATIENCE:
                status = "plato"; print(f"  [{tag}] CONVERGIU (plato de kNN-monitor). Parando."); break
        if elapsed_h >= WALLCLOCK_HOURS_PER_RUN:
            status = "timebox"; print(f"  [{tag}] TIMEBOX ({elapsed_h:.1f}h). Congelando melhor checkpoint."); break
    else:
        status = "epochs_cap"

    pd.DataFrame(hist).to_csv(OUT_DIR/f"convergencia_{tag}.csv", index=False)
    summary = dict(tag=tag, seed=pretrain_seed, aug=aug_config, backbone=BACKBONE, batch=batch,
                   status=status, best_knn_ba=round(best_ba,2), best_epoch=best_ep,
                   epochs_run=(hist[-1]["epoch"] if hist else 0),
                   final_std=(hist[-1]["std"] if hist else None),
                   wallclock_h=round((time.time()-t_start)/3600.0, 2), best_ckpt=str(best_path))
    (OUT_DIR/f"convergencia_{tag}.json").write_text(json.dumps(summary, indent=2))
    print(f"  [{tag}] FIM status={status} best_knn_ba={best_ba:.2f}@{best_ep} batch={batch}")
    return summary

def train_simclr(pretrain_seed, aug_config, tag, batch=None, min_batch=None):
    b = batch if batch is not None else BATCH
    floor = min_batch if min_batch is not None else MIN_BATCH
    while b >= floor:
        try:
            return _try_train(pretrain_seed, aug_config, tag, b)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" not in str(e).lower():
                raise                                        # nao engole erros que nao sao OOM
            torch.cuda.empty_cache()
            print(f"  [OOM] batch {b} (forward {2*b}) nao coube -> {b//2}. "
                  f"AVISO: menos negativos DEGRADA o SimCLR. REGISTRAR com data no .md.")
            b //= 2
    raise RuntimeError(f"OOM ate min_batch={floor}. Use BACKBONE='resnet18' ou DDP multi-GPU (all_gather).")

print("train_simclr (grad-ckpt + channels_last + auto-fit de batch) pronto")


## 6. Congelar + extrair features (alinhadas a `paths_100k`)

Carrega o melhor checkpoint, encoder em `eval` (checkpointing off), extrai **sem augmentation** para
toda a lista canonica -> `.npy` cacheado (via memmap na releitura, p/ nao estourar RAM do host com
varias seeds em memoria). Cai no MESMO aparato de probe/PCA da Semana 2.

In [ ]:
def cache_key(name, n):
    h = hashlib.md5(f"{name}|{BACKBONE}|{n}".encode()).hexdigest()[:10]
    return CACHE_DIR / f"{name}_n{n}_{h}.npy"

def freeze_and_extract(ckpt_path, tag, paths, ptag):
    fp = cache_key(f"simclr_{tag}_{ptag}", len(paths))
    if fp.exists():
        print(f"[cache] features {tag}/{ptag}: {fp.name}")
        return np.load(fp, mmap_mode="r")                    # memmap: economiza RAM do host
    ck = torch.load(ckpt_path, map_location=DEVICE)
    model = SimCLRNet(ck["backbone"], PROJ_DIM, grad_ckpt=False).to(DEVICE).to(memory_format=torch.channels_last)
    model.load_state_dict(ck["state_dict"])
    X = encode(model, paths).astype(np.float32)
    np.save(fp, X)
    del model; torch.cuda.empty_cache()
    print(f"[extract] {tag}/{ptag}: {X.shape} -> {fp.name}")
    return np.load(fp, mmap_mode="r")
print("freeze_and_extract pronto")


## 7. Probe + condicao PCA (identico a Semana 2, sem vazamento)

`Pipeline(StandardScaler -> [PCA] -> StandardScaler -> LogReg)`, PCA ajustada **so no fold de
treino**, refeita por `(frac, seed)`. Mesmo probe dos outros bracos. A indexacao `Xfeat_all[tr]`
sobre memmap devolve copia so das linhas usadas -> pico de RAM baixo.

In [ ]:
def make_probe(pca_dim=None):
    steps = [("sc", StandardScaler())]
    if pca_dim is not None:
        steps += [("pca", PCA(n_components=pca_dim, random_state=0)), ("sc2", StandardScaler())]
    steps += [("clf", LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1))]
    return Pipeline(steps)

def eval_indices():
    if EVAL_TARGET == "internal":
        return idx_eval_int, y_100k[idx_eval_int]
    raise RuntimeError("external_7k e TOQUE UNICO - use run_external_7k_once() (celula 14).")

def probe_on(Xfeat_all, frac, seed, pca_dim, idx_eval, y_eval):
    sub = label_subsample(frac, seed); tr = idx_train_pool[sub]
    Xtr = np.asarray(Xfeat_all[tr]); Xte = np.asarray(Xfeat_all[idx_eval])   # materializa so o necessario
    pipe = make_probe(pca_dim); pipe.fit(Xtr, y_100k[tr])
    yp = pipe.predict(Xte)
    row = dict(frac=frac, seed=seed, cond=("RAW" if pca_dim is None else f"PCA{pca_dim}"),
               n_train=len(tr), bal_acc=100*balanced_accuracy_score(y_eval, yp),
               f1_macro=100*f1_score(y_eval, yp, average="macro"))
    if pca_dim is not None:
        row["pca_evr"] = float(pipe.named_steps["pca"].explained_variance_ratio_.sum())
    return row
print("probe pronto")


## 8. `[P1]` Primeiro SimCLR (seed 0, full-aug) e a BARRA go/no-go

Prioridade 1: **um** SimCLR utilizavel. A pergunta nao e o numero do paper (interno nao e paper) -
e **o SimCLR-indomain limpa a barra do melhor classico com folga?** Se sim, o eixo SSL das H1-H3
esta vivo. Se nao, fallback §10-3 (guarda-c).

In [ ]:
tag0 = f"s0_{AUG_CONFIG}"
summ0 = train_simclr(pretrain_seed=PRETRAIN_SEEDS[0], aug_config=AUG_CONFIG, tag=tag0)
X_ssl0 = freeze_and_extract(summ0["best_ckpt"], tag0, paths_100k, "100k")
print("status de convergencia P1:", summ0["status"], "| best kNN-monitor:", summ0["best_knn_ba"],
      "| batch efetivo:", summ0["batch"])


In [ ]:
idx_ev, y_ev = eval_indices()

def load_week2_arms():
    if WEEK2_CACHE_DIR is None:
        print("[barra] WEEK2_CACHE_DIR=None -> comparacao com classico pulada. "
              "Aponte para os .npy da Semana 2 (LBP/Fractal/Phikon).")
        return {}
    d = Path(WEEK2_CACHE_DIR); arms = {}
    for name, stem in [("LBP","lbp_100k"), ("Fractal","frac_100k"), ("Phikon","phikon_100k")]:
        hits = sorted(d.glob(f"{stem}_n*.npy"))
        if hits:
            X = np.load(hits[0], mmap_mode="r")              # memmap tb p/ os bracos da Semana 2
            if X.shape[0] == N: arms[name] = X
            else: print(f"[barra] {name}: n={X.shape[0]} != {N}, ignorado.")
    print("[barra] bracos da Semana 2 carregados:", list(arms.keys()))
    return arms

arms = load_week2_arms()
arms["SimCLR_s0"] = X_ssl0

def fraction_curve(Xall, cond_pca=None):
    out = []
    for frac in LABEL_FRACTIONS:
        seeds = LABEL_SEEDS if frac < 1.0 else LABEL_SEEDS[:1]
        vals = [probe_on(Xall, frac, s, cond_pca, idx_ev, y_ev)["bal_acc"] for s in seeds]
        m = float(np.mean(vals)); sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        ci = 1.96*sd/np.sqrt(len(vals)) if len(vals) > 1 else np.nan
        out.append(dict(frac=frac, bal_acc=round(m,2), ci95=(round(ci,2) if len(vals)>1 else None)))
    return pd.DataFrame(out)

print("=== Curva de fracao (RAW, interno) - SimCLR_s0 ===")
print(fraction_curve(X_ssl0))
for name in ["LBP","Fractal","Phikon"]:
    if name in arms:
        print(f"\n=== {name} (referencia da Semana 2) ===")
        print(fraction_curve(arms[name]))


In [ ]:
if "LBP" in arms:
    ssl_curve = fraction_curve(X_ssl0).set_index("frac")["bal_acc"]
    lbp_curve = fraction_curve(arms["LBP"]).set_index("frac")["bal_acc"]
    frac_curve= fraction_curve(arms["Fractal"]).set_index("frac")["bal_acc"] if "Fractal" in arms else None
    print("\n--- BARRA go/no-go do eixo SSL (interno; NAO e numero de paper) ---")
    for frac in LABEL_FRACTIONS:
        s = ssl_curve.loc[frac]
        c = lbp_curve.loc[frac] if frac_curve is None else max(lbp_curve.loc[frac], frac_curve.loc[frac])
        print(f"  frac {int(frac*100):3d}% : SimCLR {s:5.2f}  melhor_classico {c:5.2f}  delta {s-c:+5.2f}")
    print("SimCLR ACIMA do classico com folga (esp. 1-5%) => eixo SSL vivo.")
    print("SimCLR colado/abaixo => achado reportavel + fallback §10-3 (guarda-c).")
    print("REGISTRAR a decisao (GO eixo SSL | fallback §10-3) com data no .md.")
else:
    print("Sem classico carregado -> barra nao avaliada. Aponte WEEK2_CACHE_DIR.")


## 9. `[P2]` Variancia HONESTA: seeds de PRE-TREINO (a guarda-f, operacionalizada)

**O ponto central.** A variancia que importa para o SimCLR e **across-pretrain**, nao a de
subamostragem de rotulo sobre um extrator congelado. Reporta as duas lado a lado. Se so a seed 0
completar: registrar **subcaracterizada** e nao reportar o IC within-probe como se fosse do SimCLR.

In [ ]:
ssl_feats = {PRETRAIN_SEEDS[0]: X_ssl0}
for sd in PRETRAIN_SEEDS[1:]:
    tag = f"s{sd}_{AUG_CONFIG}"
    print(f"\n>>> P2: pre-treino seed {sd} ({AUG_CONFIG}) - protegido por timebox")
    summ = train_simclr(pretrain_seed=sd, aug_config=AUG_CONFIG, tag=tag)
    ssl_feats[sd] = freeze_and_extract(summ["best_ckpt"], tag, paths_100k, "100k")

n_done = len(ssl_feats)
print(f"\nPre-treinos concluidos: {n_done} ({sorted(ssl_feats)})")
if n_done < 2:
    print("[guarda-f] SO 1 pre-treino. Variancia do SSL SUBCARACTERIZADA - declarar no paper.")


In [ ]:
def variance_decomposition(frac, cond_pca=None, probe_seed=LABEL_SEEDS[0]):
    across = np.array([probe_on(ssl_feats[sd], frac, probe_seed, cond_pca, idx_ev, y_ev)["bal_acc"]
                       for sd in sorted(ssl_feats)])                       # varia PRE-TREINO (honesta)
    sd0 = sorted(ssl_feats)[0]
    within_seeds = LABEL_SEEDS if frac < 1.0 else LABEL_SEEDS[:1]
    within = np.array([probe_on(ssl_feats[sd0], frac, s, cond_pca, idx_ev, y_ev)["bal_acc"]
                       for s in within_seeds])                             # varia PROBE (enganosa)
    def stat(a):
        m = a.mean(); sd = a.std(ddof=1) if len(a) > 1 else 0.0
        ci = 1.96*sd/np.sqrt(len(a)) if len(a) > 1 else np.nan
        return round(m,2), round(sd,3), (round(ci,2) if len(a)>1 else None), len(a)
    ma, sda, cia, na = stat(across); mw, sdw, ciw, nw = stat(within)
    return dict(frac=frac, cond=("RAW" if cond_pca is None else f"PCA{cond_pca}"),
                across_mean=ma, across_sd=sda, across_ci95=cia, n_pretrain=na,
                within_mean=mw, within_sd=sdw, within_ci95=ciw, n_probe=nw)

vd = pd.DataFrame([variance_decomposition(f) for f in [0.01, 0.05, 1.00]])
vd.to_csv(OUT_DIR/"variancia_pretrain_vs_probe.csv", index=False)
print("=== Decomposicao de variancia (RAW, interno) ===")
print(vd.to_string(index=False))
print("\nacross_ci95 (varia pre-treino) = variancia HONESTA. within_ci95 (varia so probe) sub-reporta "
      "(guarda-f). n_pretrain=1 => across_ci95=None => SUBCARACTERIZADA.")


## 10. H2 no eixo SimCLR (PCA-matched + variancia retida + d-sweep)

O SimCLR **tem headroom** (nao satura como o Phikon), entao o teste forte de H2 e aqui. Reproduz o
aparato da Semana 2: gap RAW vs PCA-matched, EVR retida em `COMMON_D`, curva desempenho-vs-`d`.

In [ ]:
def evr_at(Xall, d, frac=1.0, seed=LABEL_SEEDS[0]):
    return probe_on(Xall, frac, seed, d, idx_ev, y_ev).get("pca_evr")

print("Variancia retida (EVR) em d=COMMON_D=%d (frac=100%%):" % COMMON_D)
for sd in sorted(ssl_feats):
    print(f"  SimCLR_s{sd}: EVR={100*evr_at(ssl_feats[sd], COMMON_D):.1f}%  (raw dim {ssl_feats[sd].shape[1]})")
for name in ["LBP","Fractal","Phikon"]:
    if name in arms:
        print(f"  {name}: EVR={100*evr_at(arms[name], COMMON_D):.1f}%  (raw dim {arms[name].shape[1]})")

def gap_raw_vs_pca(Xhi, Xlo, frac):
    sub = label_subsample(frac, LABEL_SEEDS[0]); tr = idx_train_pool[sub]
    def ba(X, d):
        p = make_probe(d); p.fit(np.asarray(X[tr]), y_100k[tr])
        return 100*balanced_accuracy_score(y_ev, p.predict(np.asarray(X[idx_ev])))
    return dict(frac=frac, RAW=round(ba(Xhi,None)-ba(Xlo,None),2), PCA=round(ba(Xhi,COMMON_D)-ba(Xlo,COMMON_D),2))

if "LBP" in arms:
    print("\nGap SimCLR_s0 - LBP, RAW vs PCA%d (H2 no eixo SSL):" % COMMON_D)
    for f in [0.05, 1.00]:
        g = gap_raw_vs_pca(X_ssl0, arms["LBP"], f)
        print(f"  frac {int(f*100):3d}%: RAW {g['RAW']:+.2f}  PCA {g['PCA']:+.2f}  delta {g['PCA']-g['RAW']:+.2f}")
    print("delta>0 (gap CRESCE sob PCA) tambem no SSL => refutacao de H2 replica no eixo SSL.")


In [ ]:
sweep = []
for sd in sorted(ssl_feats):
    for frac in D_SWEEP_FRACTIONS:
        for d in D_SWEEP:
            if d > ssl_feats[sd].shape[1]: continue
            for ps in LABEL_SEEDS[:3]:
                r = probe_on(ssl_feats[sd], frac, ps, d, idx_ev, y_ev)
                r["arm"] = f"SimCLR_s{sd}"; sweep.append(r)
if sweep:
    dfs = pd.DataFrame(sweep); dfs.to_csv(OUT_DIR/"pca_sweep_simclr.csv", index=False)
    print("Desempenho-vs-d (SimCLR, bal.acc media):")
    print(dfs.groupby(["arm","frac","cond"])["bal_acc"].mean().round(2))


## 11. `[P3, cortavel]` Ablacao H3-a: remover a invariancia escala/cor do SSL

O que faz o SimCLR ser o **objeto controlavel**. Treina `AUG_CONFIG="ablated_no_scale_color"` e
compara a **vantagem sobre o classico** entre full e ablado. vantagem(full) > vantagem(ablado) =>
suporta H3-a. **Custo:** >=1 pre-treino a mais - **primeiro corte do timebox.**

In [ ]:
RUN_H3A = True   # deixe True so quando P1/P2 estiverem no bolso e sobrar timebox

if RUN_H3A:
    abl_feats = {}
    for sd in PRETRAIN_SEEDS[:max(1, len(PRETRAIN_SEEDS)-1)]:
        tag = f"s{sd}_ablated_no_scale_color"
        print(f"\n>>> H3-a: pre-treino ablado seed {sd} - protegido por timebox")
        summ = train_simclr(pretrain_seed=sd, aug_config="ablated_no_scale_color", tag=tag)
        abl_feats[sd] = freeze_and_extract(summ["best_ckpt"], tag, paths_100k, "100k")

    if "LBP" in arms:
        base_cls = arms["LBP"]
        def adv_over_lbp(Xssl, frac, ps=LABEL_SEEDS[0]):
            sub = label_subsample(frac, ps); tr = idx_train_pool[sub]
            def ba(X):
                p = make_probe(None); p.fit(np.asarray(X[tr]), y_100k[tr])
                return 100*balanced_accuracy_score(y_ev, p.predict(np.asarray(X[idx_ev])))
            return ba(Xssl) - ba(base_cls)
        print("\n=== H3-a: vantagem sobre LBP (media across-pretrain), RAW ===")
        for frac in [0.05, 1.00]:
            full_adv = np.mean([adv_over_lbp(ssl_feats[s], frac) for s in sorted(ssl_feats)])
            abl_adv  = np.mean([adv_over_lbp(abl_feats[s], frac) for s in sorted(abl_feats)])
            print(f"  frac {int(frac*100):3d}%: full {full_adv:+.2f}  ablado {abl_adv:+.2f}  "
                  f"queda {full_adv-abl_adv:+.2f}  => {'suporta H3-a' if full_adv-abl_adv>0 else 'H3-a nulo'}")
        print("Ressalva: interno, Phikon-saturado a parte; confirmar no 7k antes do paper (guarda-a).")
else:
    print("H3-a nao rodado (RUN_H3A=False). Se foi corte de timebox, registrar -> vira limitacao declarada.")


## 12. Snippet de registro (colar em `registro_semana34.md`)

Reconstruir estas semanas na escrita **sem reabrir o notebook**: numeros, status de convergencia
(inclui batch efetivo se o auto-fit reduziu), decomposicao de variancia, barra, H3-a e guardas.

In [ ]:
def emit_registro():
    L = []
    L.append("# Registro - Semanas 3-4 (WVC 2026, §11): SimCLR-indomain\n")
    L.append(f"**EVAL_TARGET:** {EVAL_TARGET} (interno = DESENVOLVIMENTO, nao numero de paper)\n")
    L.append(f"**Config:** backbone={BACKBONE} batch_alvo={BATCH} grad_ckpt={GRAD_CKPT} "
             f"epochs_cap={EPOCHS_CAP} temp={TEMP} aug={AUG_CONFIG} "
             f"alloc={os.environ.get('PYTORCH_CUDA_ALLOC_CONF')} pool="
             f"{'train_pool_80k' if PRETRAIN_ON_TRAIN_POOL_ONLY else 'full_100k'}\n")
    L.append("\n## Convergencia por pre-treino (status + batch efetivo do auto-fit)\n")
    for sd in sorted(ssl_feats):
        jf = OUT_DIR/f"convergencia_s{sd}_{AUG_CONFIG}.json"
        if jf.exists():
            s = json.loads(jf.read_text())
            L.append(f"- seed {sd} ({AUG_CONFIG}): status={s['status']} "
                     f"best_knn_ba={s['best_knn_ba']}@ep{s['best_epoch']} "
                     f"batch_efetivo={s.get('batch')} std_final={s['final_std']} wallclock={s['wallclock_h']}h\n")
    L.append("\n> Se algum batch_efetivo < batch_alvo, o auto-fit reduziu por OOM: esse SimCLR tem "
             "MENOS negativos e esta enfraquecido por MEMORIA, nao por metodo. Considerar isso na "
             "leitura da barra e declarar na limitacao.\n")
    L.append("\n## Curva de fracao - SimCLR_s0 (RAW, interno)\n")
    L.append(fraction_curve(X_ssl0).to_markdown(index=False))
    L.append("\n\n## Variancia across-pretrain vs within-probe (guarda-f)\n")
    L.append(pd.DataFrame([variance_decomposition(f) for f in [0.01,0.05,1.00]]).to_markdown(index=False))
    n_pt = len(ssl_feats)
    L.append(f"\n\n**n_pretrain = {n_pt}.** "
             + ("Variancia do SSL SUBCARACTERIZADA (1 seed) - declarar no paper.\n" if n_pt < 2
                else "across_ci95 e a variancia reportavel do SimCLR.\n"))
    L.append("\n## Guardas (nao apagar)\n")
    L.append("- (a) Numeros internos NAO sao de paper; so o 7k entra em tabela de desempenho.\n")
    L.append("- (f) Variancia do SimCLR = ACROSS-PRETRAIN. within-probe sub-reporta.\n")
    L.append("- (transdutiva) pre-treino so em train_pool; eval_int intocado. Desvio REGISTRADO do §6.\n")
    L.append("- (barra) GO do eixo SSL depende do SimCLR limpar o classico com folga; senao, fallback §10-3.\n")
    L.append("- (OOM) grad_ckpt + expandable_segments + channels_last aplicados; batch efetivo registrado acima.\n")
    L.append("- (stain) confirmar que NCT_DIR era Macenko (herdado da Semana 2).\n")
    txt = "".join(L)
    (OUT_DIR/"registro_semana34.md").write_text(txt, encoding="utf-8")
    print(txt[:1400]); print("...\n[gravado em", OUT_DIR/"registro_semana34.md","]")

emit_registro()


## 13. (So no run final) Avaliacao externa no 7k - **TOQUE UNICO**

Nao rodar no desenvolvimento. Extrai features do 7k na **mesma ordem canonica**, reusa o mesmo
probe, inclui a condicao PCA-matched (confirmar H2 fora do regime saturado - item que **bloqueia H2
no paper**, herdado da Semana 2). **Estes SIM sao numeros de paper.**

In [ ]:
def run_external_7k_once(which_ssl_seed=None):
    assert EVAL_TARGET != "internal", "Mude EVAL_TARGET p/ 'external_7k' e confirme o pre-commit da tabela."
    paths_7k, y_7k = index_folder(CRC_VAL_DIR, CLASSES)
    sd = which_ssl_seed if which_ssl_seed is not None else sorted(ssl_feats)[0]
    ck = CKPT_DIR / f"simclr_s{sd}_{AUG_CONFIG}_best.pt"
    X7_ssl = np.asarray(freeze_and_extract(ck, f"s{sd}_{AUG_CONFIG}", paths_7k, "7k"))
    X100 = np.asarray(ssl_feats[sd])
    out = []
    for cond in [None, COMMON_D]:
        for frac in LABEL_FRACTIONS:
            for seed in (LABEL_SEEDS if frac < 1.0 else LABEL_SEEDS[:1]):
                sub = label_subsample(frac, seed); tr = idx_train_pool[sub]
                p = make_probe(cond); p.fit(X100[tr], y_100k[tr])
                yp = p.predict(X7_ssl)
                out.append(dict(arm=f"SimCLR_s{sd}", frac=frac, seed=seed,
                                cond=("RAW" if cond is None else f"PCA{cond}"),
                                bal_acc=100*balanced_accuracy_score(y_7k, yp),
                                f1_macro=100*f1_score(y_7k, yp, average="macro")))
    d7 = pd.DataFrame(out); d7.to_csv(OUT_DIR/"resultados_7k_simclr_FINAL.csv", index=False)
    print("7k FINAL (SimCLR) gravado. Rodar tambem LBP/Fractal/Phikon no 7k + PCA - H2 no 7k bloqueia o paper.")
    return d7

print("run_external_7k_once() disponivel - NAO chamar no desenvolvimento.")


---
### Fecho - o que estas semanas entregam e o que NAO provam

**Entrega:** SimCLR-indomain treinado com timebox em codigo, **fixes de OOM embutidos** (grad
checkpointing + `expandable_segments` + `channels_last` + auto-fit de batch que preserva negativos
e so reduz em ultimo caso, registrando), criterio de convergencia pre-registrado com status
honesto, features congeladas alinhadas a `paths_100k`, a barra SimCLR-vs-classico, a decomposicao de
variancia across-pretrain vs within-probe (guarda-f), H2 no eixo SSL, e - se o timebox permitir - a
ablacao H3-a.

**Nao prova (e a escrita nao pode sugerir):**
- Nada em tabela de desempenho de paper - tudo aqui e interno. So o 7k (toque unico) entra.
- Se `n_pretrain < 2`, variancia do SimCLR **subcaracterizada**.
- Se o auto-fit reduziu o batch por OOM, o SimCLR esta enfraquecido por **memoria**, nao por metodo -
  isso muda a leitura da barra e vira limitacao declarada.
- Refutacao de H2 no eixo SSL e queda em H3-a **precisam de confirmacao no 7k**.
- SimCLR fraco => fallback §10-3 (achado, nao buraco).

**Itens que seguem bloqueando o paper (nao destas semanas):** H2 no 7k com PCA; decisao do Fractal
(Bouligand-Minkowski/multifractal vs. manter+citar); confirmar Macenko upstream; bibliografia do
eva. Nenhum depende do SimCLR - adianta-los em paralelo protege o cronograma.
